In [3]:

from pathlib import Path

project = Path("academic_portal")
project.mkdir(exist_ok=True)

env_file = Path(".env")

env_file.write_text(
    "PASS_MARKS=40\nFINE_PER_DAY=50\n",
    encoding="utf-8"
)

print("Project folder and configuration created!")


Project folder and configuration created!


In [4]:

import csv

students = [
    ["student_id", "name", "marks", "late_days"],
    ["101", "Ali", "75", "0"],
    ["102", "Sara", "32", "3"],
    ["103", "Ahmed", "85", "2"],
    ["104", "Ayesha", "25", "0"],
    ["105", "Usman", "110", "1"]
]

with open(project / "students.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(students)

print("Student CSV file created!")


Student CSV file created!


In [5]:

%%writefile academic_portal/exceptions.py

class InvalidStudentRecord(Exception):
    pass


Writing academic_portal/exceptions.py


In [6]:

%%writefile academic_portal/student.py

from exceptions import InvalidStudentRecord

class Student:
    def __init__(self, student_id, name, marks, late_days, pass_marks, fine_rate):
        self.student_id = student_id
        self.name = name
        self.marks = marks
        self.late_days = late_days
        self.status = "Pass" if marks >= pass_marks else "Fail"
        self.fine = late_days * fine_rate

    @classmethod
    def from_row(cls, row, pass_marks, fine_rate):
        try:
            student_id = row["student_id"].strip()
            name = row["name"].strip()
            marks = float(row["marks"])
            late_days = int(row["late_days"])

            if not student_id or not name:
                raise InvalidStudentRecord("Student ID or name is missing")

            if not 0 <= marks <= 100:
                raise InvalidStudentRecord("Marks must be between 0 and 100")

            if late_days < 0:
                raise InvalidStudentRecord("Late days cannot be negative")

            return cls(
                student_id, name, marks, late_days,
                pass_marks, fine_rate
            )

        except (ValueError, KeyError) as error:
            raise InvalidStudentRecord("Invalid student data") from error


Writing academic_portal/student.py


In [7]:

%%writefile academic_portal/main.py

import csv
import os
import asyncio
import logging
import time
from pathlib import Path
from datetime import datetime
from functools import wraps
from dotenv import load_dotenv

from exceptions import InvalidStudentRecord
from student import Student

# Project paths and configuration
BASE_DIR = Path(__file__).resolve().parent
load_dotenv(BASE_DIR.parent / ".env")

PASS_MARKS = int(os.getenv("PASS_MARKS", "40"))
FINE_PER_DAY = float(os.getenv("FINE_PER_DAY", "50"))

# Logging setup
logging.basicConfig(
    filename=BASE_DIR / "portal.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("AcademicPortal")


# Decorator to calculate execution time
def show_time(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        start = time.time()
        result = await func(*args, **kwargs)
        logger.info(
            "%s completed in %.2f seconds",
            func.__name__, time.time() - start
        )
        return result
    return wrapper


# Process and validate one student
async def process_student(row):
    await asyncio.sleep(0)

    try:
        student = Student.from_row(
            row, PASS_MARKS, FINE_PER_DAY
        )
        return student, None

    except InvalidStudentRecord as error:
        logger.error("Invalid record %s: %s", row, error)
        return None, {"Student ID": row.get("student_id", ""),
                      "Error": str(error)}


# Main async pipeline
@show_time
async def run_pipeline():
    students = []
    invalid_records = []

    input_file = BASE_DIR / "students.csv"

    with open(input_file, "r", newline="") as file:
        rows = list(csv.DictReader(file))

    results = await asyncio.gather(
        *(process_student(row) for row in rows)
    )

    for student, error in results:
        if student is not None:
            students.append(student)
        else:
            invalid_records.append(error)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    report_file = BASE_DIR / f"academic_report_{timestamp}.csv"
    error_file = BASE_DIR / f"invalid_records_{timestamp}.csv"

    # Save valid student report
    with open(report_file, "w", newline="") as file:
        fields = [
            "Student ID", "Name", "Marks",
            "Status", "Late Days", "Fine (Rs.)"
        ]

        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()

        for student in students:
            writer.writerow({
                "Student ID": student.student_id,
                "Name": student.name,
                "Marks": student.marks,
                "Status": student.status,
                "Late Days": student.late_days,
                "Fine (Rs.)": student.fine
            })

    # Save invalid records separately
    with open(error_file, "w", newline="") as file:
        writer = csv.DictWriter(
            file, fieldnames=["Student ID", "Error"]
        )
        writer.writeheader()
        writer.writerows(invalid_records)

    failed = sum(
        1 for student in students if student.status == "Fail"
    )
    total_fines = sum(student.fine for student in students)

    logger.info("Academic reports generated successfully")

    print("\n===== ACADEMIC PORTAL =====")
    print("Total records:", len(rows))
    print("Valid students:", len(students))
    print("Invalid records:", len(invalid_records))
    print("Failed students:", failed)
    print("Total late fines: Rs.", total_fines)
    print("Academic report:", report_file.name)
    print("Invalid records report:", error_file.name)
    print("Error log: portal.log")
    print("===========================")


if __name__ == "__main__":
    try:
        asyncio.run(run_pipeline())
    except Exception:
        logger.exception("Academic Portal failed")
        print("An error occurred. Check portal.log for details.")


Writing academic_portal/main.py


In [8]:

gitignore_file = Path(".gitignore")

gitignore_file.write_text(
    ".env\n.venv/\n__pycache__/\n*.pyc\n",
    encoding="utf-8"
)

print("Private configuration protected from Git tracking!")


Private configuration protected from Git tracking!


In [9]:

!python academic_portal/main.py



===== ACADEMIC PORTAL =====
Total records: 5
Valid students: 4
Invalid records: 1
Failed students: 2
Total late fines: Rs. 250.0
Academic report: academic_report_20261020_175424.csv
Invalid records report: invalid_records_20261020_175424.csv
Error log: portal.log


In [10]:

import pandas as pd
import glob

report_files = glob.glob(
    "academic_portal/academic_report_*.csv"
)

latest_report = max(report_files)

display(pd.read_csv(latest_report))


,Student ID,Name,Marks,Status,Late Days,Fine (Rs.)
0,101,Ali,75.0,Pass,0,0.0
1,102,Sara,32.0,Fail,3,150.0
2,103,Ahmed,85.0,Pass,2,100.0
3,104,Ayesha,25.0,Fail,0,0.0
